# Stability MAE and R2 statistics


In [8]:
import pandas as pd
from pathlib import Path


## Configuration


In [9]:
BASELINE_OUT_ROOT = "comp"
STABILITY_ROOT = "stability"
SPACE_NAME = "output_proj"

L2_NORM = True
METRIC = "euclidean"
CLUSTER_SELECTION_METHOD = "eom"
CLUSTER_SELECTION_EPSILON = 0.0
N_PERMUTATIONS = 5000
RANDOM_STATE = 0

STABILITY_PARAM_CONFIGS = [
    {
        "min_cluster_size": 6,
        "min_samples": 5,
    },
    {
        "min_cluster_size": 5,
        "min_samples": 6,
    },
    {
        "min_cluster_size": 4,
        "min_samples": 5,
    },
    {
        "min_cluster_size": 5,
        "min_samples": 4,
    },
    {
        "min_cluster_size": 6,
        "min_samples": 6,
    },
    {
        "min_cluster_size": 4,
        "min_samples": 4,
    },
]

MODEL_RUN_CONFIGS = [
    {
        "label": "mistral",
        "model_name": "mistralai/Mistral-7B-v0.1",
        "tokenizer_path": "mistralai/Mistral-7B-v0.1/tokenizer",
        "full_pca_dim": 4096,
        "candidate_dims": [5, 142, 997, 2084, 3031, 3459, 3938, 4088, 4096],
    },
    {
        "label": "mixtral",
        "model_name": "mistralai/Mixtral-8x7B-v0.1",
        "tokenizer_path": "mistralai/Mixtral-8x7B-v0.1/tokenizer",
        "full_pca_dim": 4096,
        "candidate_dims": [8, 158, 1111, 2156, 3052, 3457, 3957, 4093, 4096],
    },
    {
        "label": "gpt-oss",
        "model_name": "gpt-oss",
        "tokenizer_path": "gpt-oss/tokenizer",
        "full_pca_dim": 2880,
        "candidate_dims": [6, 182, 466, 739, 1591, 2264, 2532, 2868, 2880],
    },
]


## Helpers


In [10]:
def make_stability_param_tag(param_config):
    return f"mcs={int(param_config['min_cluster_size'])}_ms={int(param_config['min_samples'])}"


def stability_out_root_for_config(param_config):
    return Path(STABILITY_ROOT) / make_stability_param_tag(param_config)


## Per-Config MAE and R2


In [11]:
def compute_curve_r2(baseline_values, stability_values):
    baseline_values = pd.Series(baseline_values, dtype=float).to_numpy()
    stability_values = pd.Series(stability_values, dtype=float).to_numpy()
    ss_res = float(((stability_values - baseline_values) ** 2).sum())
    ss_tot = float(((baseline_values - baseline_values.mean()) ** 2).sum())
    if ss_tot == 0.0:
        return float("nan")
    return 1.0 - ss_res / ss_tot


def compute_stability_curve_mae_report():
    rows = []

    for param_config in STABILITY_PARAM_CONFIGS:
        param_tag = make_stability_param_tag(param_config)
        stability_out_root = stability_out_root_for_config(param_config)

        print(f"\n=== {param_tag} ===")

        for model_config in MODEL_RUN_CONFIGS:
            model_name = model_config["model_name"]
            baseline_model_dir = Path(BASELINE_OUT_ROOT) / model_name / SPACE_NAME
            stability_model_dir = stability_out_root / model_name / SPACE_NAME

            baseline_morph_path = baseline_model_dir / "kondrak_global_summary.csv"
            baseline_script_path = baseline_model_dir / "script_entropy_summary.csv"
            stability_morph_path = stability_model_dir / "kondrak_global_summary.csv"
            stability_script_path = stability_model_dir / "script_entropy_summary.csv"

            required_paths = [
                baseline_morph_path,
                baseline_script_path,
                stability_morph_path,
                stability_script_path,
            ]
            missing_paths = [str(p) for p in required_paths if not p.exists()]
            if missing_paths:
                print(f"{model_name}: missing files")
                for missing_path in missing_paths:
                    print(f"  {missing_path}")
                continue

            baseline_morph = pd.read_csv(baseline_morph_path)
            baseline_script = pd.read_csv(baseline_script_path)
            stability_morph = pd.read_csv(stability_morph_path)
            stability_script = pd.read_csv(stability_script_path)

            morph_aligned = (
                baseline_morph[["pca_dim", "global_mean"]]
                .merge(
                    stability_morph[["pca_dim", "global_mean"]],
                    on="pca_dim",
                    how="inner",
                    suffixes=("_baseline", "_stability"),
                )
                .sort_values("pca_dim")
                .reset_index(drop=True)
            )
            script_aligned = (
                baseline_script[["pca_dim", "mean_H"]]
                .merge(
                    stability_script[["pca_dim", "mean_H"]],
                    on="pca_dim",
                    how="inner",
                    suffixes=("_baseline", "_stability"),
                )
                .sort_values("pca_dim")
                .reset_index(drop=True)
            )

            if morph_aligned.empty:
                raise ValueError(f"No overlapping morphology pca_dim values for {param_tag} | {model_name}")
            if script_aligned.empty:
                raise ValueError(f"No overlapping script pca_dim values for {param_tag} | {model_name}")

            morph_baseline_values = morph_aligned["global_mean_baseline"].to_numpy(dtype=float)
            morph_stability_values = morph_aligned["global_mean_stability"].to_numpy(dtype=float)
            script_baseline_values = script_aligned["mean_H_baseline"].to_numpy(dtype=float)
            script_stability_values = script_aligned["mean_H_stability"].to_numpy(dtype=float)

            morph_abs_errors = abs(morph_stability_values - morph_baseline_values)
            script_abs_errors = abs(script_stability_values - script_baseline_values)

            morph_mean_mae = float(morph_abs_errors.mean())
            script_mean_mae = float(script_abs_errors.mean())
            joint_mean_mae = float(pd.Series([*morph_abs_errors, *script_abs_errors]).mean())

            joint_baseline_values = pd.Series([*morph_baseline_values, *script_baseline_values], dtype=float).to_numpy()
            joint_stability_values = pd.Series([*morph_stability_values, *script_stability_values], dtype=float).to_numpy()
            morph_mean_r2 = compute_curve_r2(morph_baseline_values, morph_stability_values)
            script_mean_r2 = compute_curve_r2(script_baseline_values, script_stability_values)
            joint_mean_r2 = compute_curve_r2(joint_baseline_values, joint_stability_values)

            print(
                f"{model_name}: "
                f"morph_mean_mae={morph_mean_mae:.6g}, "
                f"script_mean_mae={script_mean_mae:.6g}, "
                f"joint_mean_mae={joint_mean_mae:.6g}, "
                f"morph_mean_r2={morph_mean_r2:.6g}, "
                f"script_mean_r2={script_mean_r2:.6g}, "
                f"joint_mean_r2={joint_mean_r2:.6g}"
            )

            rows.append({
                "param_tag": param_tag,
                "model_name": model_name,
                "min_cluster_size": int(param_config["min_cluster_size"]),
                "min_samples": int(param_config["min_samples"]),
                "n_dims_morph": int(len(morph_aligned)),
                "n_dims_script": int(len(script_aligned)),
                "morph_mean_mae": morph_mean_mae,
                "script_mean_mae": script_mean_mae,
                "joint_mean_mae": joint_mean_mae,
                "morph_mean_r2": morph_mean_r2,
                "script_mean_r2": script_mean_r2,
                "joint_mean_r2": joint_mean_r2,
            })

    return pd.DataFrame(rows)


In [12]:
stability_curve_mae_df = compute_stability_curve_mae_report()
stability_curve_mae_df


=== mcs=6_ms=5 ===
mistralai/Mistral-7B-v0.1: morph_mean_mae=0.0123741, script_mean_mae=0.0109294, joint_mean_mae=0.0116517, morph_mean_r2=0.988354, script_mean_r2=0.955855, joint_mean_r2=0.995784
mistralai/Mixtral-8x7B-v0.1: morph_mean_mae=0.0107033, script_mean_mae=0.0101888, joint_mean_mae=0.010446, morph_mean_r2=0.990404, script_mean_r2=0.874338, joint_mean_r2=0.995862
gpt-oss: morph_mean_mae=0.0114161, script_mean_mae=0.0157877, joint_mean_mae=0.0136019, morph_mean_r2=0.983571, script_mean_r2=0.958557, joint_mean_r2=0.990403

=== mcs=5_ms=6 ===
mistralai/Mistral-7B-v0.1: morph_mean_mae=0.00848919, script_mean_mae=0.0102262, joint_mean_mae=0.00935769, morph_mean_r2=0.992048, script_mean_r2=0.905874, joint_mean_r2=0.993948
mistralai/Mixtral-8x7B-v0.1: morph_mean_mae=0.00610015, script_mean_mae=0.00691848, joint_mean_mae=0.00650931, morph_mean_r2=0.992476, script_mean_r2=0.944731, joint_mean_r2=0.997547
gpt-oss: morph_mean_mae=0.0117605, script_mean_mae=0.0178612, joint_mean_mae=0.0

,param_tag,model_name,min_cluster_size,min_samples,n_dims_morph,n_dims_script,morph_mean_mae,script_mean_mae,joint_mean_mae,morph_mean_r2,script_mean_r2,joint_mean_r2
0,mcs=6_ms=5,mistralai/Mistral-7B-v0.1,6,5,9,9,0.012374,0.010929,0.011652,0.988354,0.955855,0.995784
1,mcs=6_ms=5,mistralai/Mixtral-8x7B-v0.1,6,5,9,9,0.010703,0.010189,0.010446,0.990404,0.874338,0.995862
2,mcs=6_ms=5,gpt-oss,6,5,9,9,0.011416,0.015788,0.013602,0.983571,0.958557,0.990403
3,mcs=5_ms=6,mistralai/Mistral-7B-v0.1,5,6,9,9,0.008489,0.010226,0.009358,0.992048,0.905874,0.993948
4,mcs=5_ms=6,mistralai/Mixtral-8x7B-v0.1,5,6,9,9,0.006100,0.006918,0.006509,0.992476,0.944731,0.997547
5,mcs=5_ms=6,gpt-oss,5,6,9,9,0.011760,0.017861,0.014811,0.984945,0.956357,0.990309
6,mcs=4_ms=5,mistralai/Mistral-7B-v0.1,4,5,9,9,0.011954,0.011184,0.011569,0.988617,0.963328,0.996200
7,mcs=4_ms=5,mistralai/Mixtral-8x7B-v0.1,4,5,9,9,0.012121,0.014565,0.013343,0.987297,0.811341,0.994114
8,mcs=4_ms=5,gpt-oss,4,5,9,9,0.011367,0.014185,0.012776,0.984663,0.972358,0.992788
9,mcs=5_ms=4,mistralai/Mistral-7B-v0.1,5,4,9,9,0.007467,0.005614,0.006541,0.994550,0.986938,0.998404


## Mean Across Parameter Configs


In [13]:
def compute_stability_metric_means_by_model(stability_metric_df):
    if stability_metric_df.empty:
        return pd.DataFrame()

    value_cols = [
        "morph_mean_mae",
        "script_mean_mae",
        "joint_mean_mae",
        "morph_mean_r2",
        "script_mean_r2",
        "joint_mean_r2",
    ]
    available_value_cols = [col for col in value_cols if col in stability_metric_df.columns]
    if not available_value_cols:
        raise ValueError("No stability metric columns are available for averaging.")

    mean_df = (
        stability_metric_df
        .groupby("model_name", as_index=False)[available_value_cols]
        .mean()
    )

    count_df = (
        stability_metric_df
        .groupby("model_name", as_index=False)
        .size()
        .rename(columns={"size": "n_param_configs"})
    )

    mean_df = count_df.merge(mean_df, on="model_name", how="inner")
    return mean_df



In [14]:
stability_metric_mean_by_model_df = compute_stability_metric_means_by_model(
    stability_curve_mae_df
)
stability_metric_mean_by_model_df

,model_name,n_param_configs,morph_mean_mae,script_mean_mae,joint_mean_mae,morph_mean_r2,script_mean_r2,joint_mean_r2
0,gpt-oss,6,0.016479,0.022307,0.019393,0.965502,0.923453,0.981504
1,mistralai/Mistral-7B-v0.1,6,0.012630,0.011038,0.011834,0.984458,0.915122,0.993086
2,mistralai/Mixtral-8x7B-v0.1,6,0.012999,0.010216,0.011608,0.981324,0.878134,0.994190


## Length-Bucket Morphology vs Full


In [15]:
LENGTH_BUCKET_MODEL_CONFIGS = [
    {
        "label": "mistral",
        "baseline_model_name": "mistralai/Mistral-7B-v0.1",
        "length_bucket_model_name": "permutation/mistralai/Mistral-7B-v0.1_length_bucket_rand",
    },
    {
        "label": "mixtral",
        "baseline_model_name": "mistralai/Mixtral-8x7B-v0.1",
        "length_bucket_model_name": "permutation/mistralai/Mixtral-8x7B-v0.1_length_bucket_rand",
    },
    {
        "label": "gpt-oss",
        "baseline_model_name": "gpt-oss",
        "length_bucket_model_name": "permutation/gpt-oss_length_bucket_rand",
    },
]


def compute_morph_curve_mae_r2(baseline_df, control_df):
    aligned = (
        baseline_df[["pca_dim", "global_mean"]]
        .merge(
            control_df[["pca_dim", "global_mean"]],
            on="pca_dim",
            how="inner",
            suffixes=("_baseline", "_control"),
        )
        .sort_values("pca_dim")
        .reset_index(drop=True)
    )
    if aligned.empty:
        raise ValueError("No overlapping pca_dim values for morphology MAE/R2.")

    baseline_values = aligned["global_mean_baseline"].to_numpy(dtype=float)
    control_values = aligned["global_mean_control"].to_numpy(dtype=float)
    abs_errors = abs(control_values - baseline_values)
    return {
        "n_dims": int(len(aligned)),
        "morph_mean_mae": float(abs_errors.mean()),
        "morph_mean_r2": compute_curve_r2(baseline_values, control_values),
    }


def compute_length_bucket_morph_mae_r2_report():
    rows = []

    for model_config in LENGTH_BUCKET_MODEL_CONFIGS:
        baseline_model_name = model_config["baseline_model_name"]
        length_bucket_model_name = model_config["length_bucket_model_name"]
        baseline_path = Path(BASELINE_OUT_ROOT) / baseline_model_name / SPACE_NAME / "kondrak_global_summary.csv"
        length_bucket_dir = Path(BASELINE_OUT_ROOT) / length_bucket_model_name / SPACE_NAME
        mean_path = length_bucket_dir / "kondrak_global_summary.csv"
        all_seed_path = length_bucket_dir / "kondrak_global_summary_all_seeds.csv"

        print(f"\n=== {baseline_model_name} | length_bucket mean ===")
        if not baseline_path.exists():
            print(f"Missing baseline morphology summary: {baseline_path}")
            continue
        if not mean_path.exists():
            print(f"Missing length-bucket morphology summary: {mean_path}")
            continue

        baseline_df = pd.read_csv(baseline_path)
        mean_df = pd.read_csv(mean_path)
        mean_stats = compute_morph_curve_mae_r2(baseline_df, mean_df)
        print(
            f"mean: "
            f"morph_mean_mae={mean_stats['morph_mean_mae']:.6g}, "
            f"morph_mean_r2={mean_stats['morph_mean_r2']:.6g}"
        )
        rows.append({
            "baseline_model_name": baseline_model_name,
            "length_bucket_model_name": length_bucket_model_name,
            "result_type": "mean",
            "perm_seed": pd.NA,
            **mean_stats,
        })

        if not all_seed_path.exists():
            print(f"Missing all-seed length-bucket morphology summary: {all_seed_path}")
            continue

        all_seed_df = pd.read_csv(all_seed_path)
        if "perm_seed" not in all_seed_df.columns:
            raise ValueError(f"{all_seed_path} missing perm_seed column")

        for perm_seed, seed_df in all_seed_df.groupby("perm_seed", sort=True):
            seed_stats = compute_morph_curve_mae_r2(baseline_df, seed_df)
            rows.append({
                "baseline_model_name": baseline_model_name,
                "length_bucket_model_name": length_bucket_model_name,
                "result_type": "seed",
                "perm_seed": int(perm_seed),
                **seed_stats,
            })

    return pd.DataFrame(rows)


In [16]:
length_bucket_morph_mae_r2_df = compute_length_bucket_morph_mae_r2_report()
length_bucket_morph_mae_r2_df



=== mistralai/Mistral-7B-v0.1 | length_bucket mean ===
mean: morph_mean_mae=0.351433, morph_mean_r2=-8.7698

=== mistralai/Mixtral-8x7B-v0.1 | length_bucket mean ===
mean: morph_mean_mae=0.33451, morph_mean_r2=-8.41552

=== gpt-oss | length_bucket mean ===
mean: morph_mean_mae=0.38963, morph_mean_r2=-15.6176


,baseline_model_name,length_bucket_model_name,result_type,perm_seed,n_dims,morph_mean_mae,morph_mean_r2
0,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,mean,<NA>,9,0.351433,-8.769802
1,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,86939546,9,0.351235,-8.770360
2,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,556019485,9,0.352147,-8.794008
3,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,827307999,9,0.350835,-8.746185
4,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,903170602,9,0.351030,-8.754421
5,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,1043521778,9,0.349950,-8.710241
6,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,1097954097,9,0.352278,-8.792262
7,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,1627694678,9,0.350924,-8.749069
8,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,1813382118,9,0.352128,-8.801675
9,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_length_b...,seed,1911784257,9,0.352029,-8.799362


In [17]:
length_bucket_morph_mae_r2_mean_df = (
    length_bucket_morph_mae_r2_df
    .loc[length_bucket_morph_mae_r2_df["result_type"] == "seed"]
    .groupby("baseline_model_name", as_index=False)
    .agg(
        n_seeds=("perm_seed", "count"),
        morph_mean_mae_mean=("morph_mean_mae", "mean"),
        morph_mean_mae_std=("morph_mean_mae", "std"),
        morph_mean_r2_mean=("morph_mean_r2", "mean"),
        morph_mean_r2_std=("morph_mean_r2", "std"),
    )
)
length_bucket_morph_mae_r2_mean_df


,baseline_model_name,n_seeds,morph_mean_mae_mean,morph_mean_mae_std,morph_mean_r2_mean,morph_mean_r2_std
0,gpt-oss,10,0.389630,0.000212,-15.617659,0.017754
1,mistralai/Mistral-7B-v0.1,10,0.351433,0.000760,-8.770097,0.029694
2,mistralai/Mixtral-8x7B-v0.1,10,0.334510,0.000417,-8.415761,0.013174
